# Reference 4 — SQL, HTTP, CLI, and Performance

A notebook of **six fully worked reference answers**. Read the reasoning, run the code, test alternate inputs, and then try to solve the same task without looking.

**Level:** Intermediate · **Mode:** Worked reference solutions · **Prerequisite:** Complete related numbered lessons first.

**Instructions:** Write clear functions or short programs, not just expressions. State assumptions, test at least one ordinary case and one edge case, and avoid external network calls. Some exercises have multiple valid designs.

## 01. SQLite CRUD

Create schema and perform inserts, updates, and deletes using safe placeholders.

**Acceptance criteria:** Operations are parameterized and connection is closed.

**Hint:** Use bound values, not SQL string interpolation.

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)")
    db.execute("INSERT INTO tasks (title) VALUES (?)", ("Read notebook",))
    db.execute("UPDATE tasks SET done = ? WHERE id = ?", (1, 1))
    assert db.execute("SELECT done FROM tasks WHERE id = ?", (1,)).fetchone() == (1,)
    db.execute("DELETE FROM tasks WHERE id = ?", (1,))
    assert db.execute("SELECT COUNT(*) FROM tasks").fetchone()[0] == 0
finally:
    db.close()

## 02. Rollback on invalid transfer

Check that a failed constraint causes the transaction to roll back.

**Acceptance criteria:** No partial update remains.

**Hint:** Transaction context manager rolls back on exceptions.

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER CHECK(balance >= 0))")
    db.executemany("INSERT INTO accounts VALUES (?, ?)", [(1, 100), (2, 0)])
    db.commit()
    try:
        with db:
            db.execute("UPDATE accounts SET balance=balance-200 WHERE id=1")
            db.execute("UPDATE accounts SET balance=balance+200 WHERE id=2")
    except sqlite3.IntegrityError:
        pass
    assert db.execute("SELECT balance FROM accounts WHERE id=1").fetchone()[0] == 100
finally:
    db.close()

## 03. Offline HTTP opener

Accept a callable opener and parse a fake JSON response without touching the network.

**Acceptance criteria:** No live HTTP call.

**Hint:** Inject a fake boundary.

In [ ]:
from urllib.request import Request
from io import BytesIO
import json

def get_json(url, opener):
    with opener(Request(url, method="GET"), timeout=3) as response:
        return json.loads(response.read().decode("utf-8"))

def fake_open(request, timeout):
    assert request.get_method() == "GET" and timeout == 3
    return BytesIO(b'{"items": [1, 2, 3]}')

assert get_json("https://example.invalid/items", fake_open)["items"] == [1, 2, 3]

## 04. Pagination loop detection

Bound pagination and reject repeated page identifiers.

**Acceptance criteria:** Stops cleanly; no unbounded request loop.

**Hint:** Visited tokens protect against cycles.

In [ ]:
def collect(get_page, first=1, max_pages=10):
    visited, values, page = set(), [], first
    while page is not None:
        if page in visited or len(visited) >= max_pages:
            raise RuntimeError("pagination limit or loop")
        visited.add(page)
        result = get_page(page)
        values.extend(result["items"])
        page = result.get("next")
    return values

pages = {1: {"items": ["A"], "next": 2}, 2: {"items": ["B"], "next": None}}
assert collect(pages.__getitem__) == ["A", "B"]

## 05. CLI parser with injected arguments

Use explicit argument values so the example runs inside a Jupyter kernel.

**Acceptance criteria:** Pure repeatable CLI behavior.

**Hint:** Avoid parsing sys.argv in notebooks.

In [ ]:
import argparse
def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("value", type=int)
    parser.add_argument("--double", action="store_true")
    options = parser.parse_args(argv)
    return options.value * (2 if options.double else 1)

assert main(["4", "--double"]) == 8
assert main(["7"]) == 7

## 06. Profiling and performance hypotheses

Compare list and set membership by repeated equivalent lookups; don't promise fixed speedups across environments.

**Acceptance criteria:** Measured evidence, no hard-coded performance requirement.

**Hint:** Set construction cost and input distribution matter.

In [ ]:
import timeit
values = list(range(3000))
set_values = set(values)
list_result = timeit.timeit(lambda: 2999 in values, number=100)
set_result = timeit.timeit(lambda: 2999 in set_values, number=100)
print("list membership:", round(list_result, 6))
print("set membership:", round(set_result, 6))
assert (2999 in values) == (2999 in set_values)

## Reflection

- Which solution uses a reusable function rather than hidden notebook state?
- Have you tested normal, boundary, and invalid cases?
- What is the expected complexity or important design trade-off?
- Can you restart the kernel and run all cells cleanly?

**Reference notebooks:** See `../solutions/` for a selection of worked examples. The solutions deliberately do not cover every challenge; explain and test the remaining problems independently.